In [2]:
import hats
import numpy as np
from dask.distributed import Client
from hats_import import CollectionArguments, pipeline_with_client, pipeline, VerificationArguments, ImportArguments
from pathlib import Path
from hats_import.catalog.file_readers import CsvReader
from astropy.io import ascii
import lsdb
import pandas as pd

hats.__version__

'0.8.1'

In [ ]:
single_file = Path("/epyc/data3/hats/raw/desi_z/zall-pix-iron.fits")

In [ ]:
args = (
    CollectionArguments(
        completion_email_address="delucchi@andrew.cmu.edu",
        output_artifact_name="desi_dr1_zcat",
        output_path="/epyc/data3/hats/catalogs/v06",
        progress_bar=True,
        simple_progress_bar=True,
    )
    .catalog(
        output_artifact_name="desi_dr1_zcat",
        input_file_list=[single_file],
        file_reader="fits",
        ra_column='TARGET_RA',
        dec_column='TARGET_DEC',
        expected_total_rows=28_425_963,
        pixel_threshold=1_000_000,
        highest_healpix_order=8,
        skymap_alt_orders=[2, 4, 6],
        row_group_kwargs={"num_rows": 200_000},
    )
    .add_margin(margin_threshold=5.0, is_default=True)
)

In [ ]:
with Client(
    local_directory="/epyc/data3/hats/tmp/",
    n_workers=2,
    threads_per_worker=1,
) as client:
    pipeline_with_client(args, client)

In [3]:
args = VerificationArguments(
    input_catalog_path="/epyc/data3/hats/catalogs/v06/desi_dr1_zcat",
    output_path="./verification/desi_dr1_zcat",
)
pipeline(args)

Loading dataset and schema.

Starting: Test hats.io.validation.is_valid_collection.
Validating collection at path /epyc/data3/hats/catalogs/v06/desi_dr1_zcat ... 
Validating catalog at path /epyc/data3/hats/catalogs/v06/desi_dr1_zcat/desi_dr1_zcat ... 
Found 75 partitions.
Approximate coverage is 91.70 % of the sky.
Validating catalog at path /epyc/data3/hats/catalogs/v06/desi_dr1_zcat/desi_dr1_zcat_5arcs ... 
Found 75 partitions.
Approximate coverage is 91.70 % of the sky.
Result: PASSED

Starting: Test that files in _metadata match the data files on disk.
Result: PASSED

Starting: Test that number of rows are equal.
	file footers vs catalog properties
	file footers vs _metadata
Result: PASSED

Starting: Test that schemas are equal, excluding metadata.
	_common_metadata vs truth
	_metadata vs truth
	file footers vs truth
Result: PASSED

Verifier results written to verification/desi_dr1_zcat/verifier_results.csv
Elapsed time (seconds): 0.21
